# GwenLand glcuda Wave 131 - production fused SwiGLU and ceilings

Counterbalanced production A/B plus reproduced glbench prefill/decode ceilings on Tesla T4.


In [ ]:
import base64
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import statistics
import subprocess
import traceback
import urllib.request
import zipfile

BUILD = "wave131-fused-swiglu-production-ceiling-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "3e2622b7137f014f11b6892000b5c7be74b67c5f"
SOURCE_REV = "3e2622b7137f014f11b6892000b5c7be74b67c5f"
PATCH_SHA256 = "b412d86b866839983730e8cfd19cd78355b390cf83a82302dad18cc87150df9a"
PATCH_GZIP_B64 = """H4sIAF5Kt2oC/+08aXPbyJXf9SvaTJUXjECI4E1q5IntUVxORpZjyZlUaVxwA2iKiECABkAdkVS1P2J/4f6Sfa8PoHHw8CSpra1alcsS+3iv+/Xrdzf9YD4nnc51kBF6dB16a58esXu6XIUsPbqjt8zu9Z1vEyeK0yxOmJWkxN1v3EEQ+eye2P5g7E/G/cnAHffG+K/vdb2h6/bmdrc79bzpdGDbtGtZbOSOJ8Opy/pDZs/t3oj1bTqa9H3mjbt0PBr2Ju7Eno4JzBsNBgedTmffNR8cHh7uve4//IF0RmNzRA7h/zGBj/OILGkQGW3SeUU+sXQdZj8YbZO8ie9/8B8ikmb+bMaSJE5ms1P89eoVeTwg6sdlkbdY0uTG8dmcJc58HjkJSwN/TcMZuYiXzMiSNWubTVOKtcmhcxqmG8Yi4GuaMWe9ctKMejfMr0w6rE+K7JEzX6fMd9K74DpcN6F5bh+LP65XayuIgsxo/6i1hDH1nWXss9B4yX9hr7+Ts/p2CbGzSmJ/7WVBHG1hs62TDiJ2R+ZByAguQzGKYMTunj+WRUf9oWePpnN3avtj2qf+eDzsd+fQOHbtUW/aH0x7bDTlHHjks9ujaB2G23hs+6KR4bpmlxzapj0eAsMdHB4dvSC/wEwCU0kQdWCCx9KUFBPJ66M3ZB4nclhv+h8p4TjIxV3w7ufP5IYlEQstDksAfBNnC0KTZQp9bEWyBSMJy4CxYZKE0ic0kh8mE7Ki2SIlLKJuyHyLXMKEOAofBDSPrlMaEthR5i1IkJK7BQOQib7GVcLgNEJAswopbIDjlJxJkFOP1isB7d3p2RlZhetULf/bmkZZ8A+ABwgW+i6BtF7WSRd0xeSOQ7oGboatHhzCZwJHgNdlxqJr2JuTJTTIZrPHd+EpbzDJ+wiu4ftotc6ej/MpcGp8EP7xNo7mwbVJxCcxrRiKhzGbveO/Rd8xovbiKM3Ix0/nZx8vnc8f3l/OyMs0S8gJaZ0xmq4TBtsHOvksY8kSblGaBR5JH9KMLfnJLlcZUBVIBtz0YJHTeyBaEJFFfEey+IYBF9CEhiELgSMydg2kWdIsCe7JEgRSsAoDj3KqwypTAvRJgDRLtoyTB5NcsigFZnkLhElNfsjz4D4nHblmcOezBLC2jtVO/vL59afL08uLGQCEg4B92N2885fXn84+f7xwPp5+cuDPYswwH3L6t4+nby9Pf3IkSS7P/3z6QYPWGww43UC6BngWRuCnQLKrdb/3hUva4phAmnLJVW/BH04ch8+G/6wsdm6ZZ+jSbknvHRANDh8Jw2ytD6i/YgnN1ihhu1ZX74pXzs2MDKptKxw4HWqtwOCMZs4K7kqYPQACS0dhWcXCZzNQAhQOzGiLAc8Hh8+SDEvmBzQylrC9e1jlX5n3w3w0eMWJAX+oPd+nFpxl5rgPxhM1iftEqAWskQU0dLzlynDb1jq6S+jKaIPcllOuYFbIUIcdkd6XAmeyjhyQCYYYJ64MnILO+XIn2vHIFg/4KPDhGs+IG8ehbI0T6oXQBAOhRdeZsIddSlNbhZUy2ONGPWXk2FHb8Hkhy0gMxDtREAIku5FzVzsfGcxxoCUunWQL8uJkA8+Spyc+PGc0i6ZOCveNATlh1pXY8hedKUGyrpOIwM4MkNJwUV8YRSf+tMQksgxS6PUWMxTNJ4/PpLQobFB/zX58BuG3Yl7G/JOrx+cvLbMMEnaVE4U8klb+oUVgJuhzbFQiH9oq02sUaejPSVDt45vRGtsa+4O4igtefBa/zm8MgZCrCGeZtrWL8P3WVs4AeHtocp0CF4iR0e1shg1G20pvgpVhtzV24eYKDMUB2oojdp9fUN4Q3zhxYrRA5V0Da+9Q6uTs/KfTn8kVfXKfoBUNki+tEpdy80Rglbi0zoTdsiTFfs4YLH1h4HhkOp8BuZAY3EZrua22PlHicuLI22e2HC5glKCgJjrRVZklBJwxKdEOKA3mH4zUpcVshgrb8bgKNXR9ql+PlOW26aDXbJjuMpYP9zCWNw/dbCtvnrPDVD5UpvJhzVQ+3Goq5xSFewXUxHGAE3odbwFEfylORBdeKIOENN8stL5LGCnzI47AXPIy4ifBPJsRkD6A4PG5JmkU/kp781L2lAplzvIkZxVW1mwGOjy/KV6Nvt5O+gop5QgdIQAUCuJlSUMUw2Ho74uJ2gZyWai1zYMkbZQcwmxUIK9ZhDYHGGBRLOR7q7SNdLHO/PguMoqribZ+Id3BLrzifGcS5NeS6sGRDo4AX6Zip+nDOGNI7f8SeM4kSAGzQGLK1eYr06R3flzAjCWxoyEQwkSJdyBjEzZcfSMinJ2xkHG7VNyK8tqtvFenN++BHTjgakTGU/ZEch1THSVPJvezlLdSYF1H9JYGIfo/rdLiVglwbxjVrtGV1AwdJfbJ4+OvrZJS/bU1e3w2f20tQFA5BXmweWZNsQd2ursjW8meEfZI/uQYOILnva9sganSkZMBGD2jDQMKIURTbp/+Hj1uMKHBxNw5uW4wHJc5GITyNefzAhRvSqs8vOEsyufBp4rTiOiSAY1+bT0+/9oC2qkFaoT2wM3KD8p9yFgKOoj6qmVJvXQDlbluQ1wWotnYuYEoxQC+go29xZKkpY+M3G1vHI8L3jayXb/gSmGVJFFFn4AJt0GAR6hl0K7ShZT6nzd9qRs8Po7XhuS/uIQryWWwZGAwyB5lKz0WQHJTVy7iWVew0W0sveTKdLClNCOZtkrTUB/loZITdMyUlUNX1AuyB0M5y3AHelUbqZDae0/FC/BtDT4dXzMK8nxURdAbqzgNcEMm+g9O4RThPE4ormidIANF1wa7Yr3kmgdubeUe7SWuSzga5bYCBSMFjUtTqkj5LS38lHKnOo+GKbkXU5lx/FtkAw/VCeFQMIguIuRRqPuvSK4+ww710Y3ivlGg/2ZhzsVvvtSGTrnihp6cX+pdsI8mPBsUx1blsUs/bJ1Y0w1VGSUNj52clXdaq3W6MDRf83hfTlOMthNCk4lUCDgBxBERHvTNRKgnh+6FccSMkjeXL742q9hWZVoDu2tsvl6CF/OwB5/nTCtuRuqswBETTM6bK9up6s7yssu9+dQyyxeTyu3pCvzE9Wr/y9F4KZTsLAXsJOGrnF3Zm9ZT3ZjWtcEDq7P9Zuh7g9iyjCqbbR1cumS601rT+ELNY3SmKaeTJt6RyDWkR+BxlVI39T6ZjHHtiT8aul3fnXTnozGbT3pDTLfYbDr1xt6wzwbuaExHljUf9QYDbzJy3cl41PUHtm9Ph+OhPXC77px1h35/2J/43rQxK9iAv5SlaejHZEyv1zWn5BB/2X1MAK7WLlhSyRoc4z/z4RcsUxk+vF7D6Yycr5DrfviFfzyL/XXIXsn02dHREXZ3QCOr1Mpd4LPOdRL45IM9goO7DlJQ0B2ULwzzKcUVL7BMJiUsk0mOpdOExe5NRX6kg0kbAlfJC2gY/EPcDhKh6UNSMK4xnkjcB0zOFJC0aBbYAeCtkhxuv2sJxvgujFaR0FrSB4mYBGjfFNAwP1THjHZOkMnMD1kFEZphPAGkUQeQl8gDn3P65PDVEmYqJdWUkdKSUdg7j8Mwvguia5maKqBtzlGlMBzc/JwCRZIKGGgFtyeLl5j8iTduHdN7Cwq8lqbIDCrPxtN3aNlFjLz9/NNrHrFh95k8kXqE6jXH9IaH53Nm/CVP1QmPN13E69DHoM+toEjSG46IAf91kviuLQgSRMCi1CfxnHNKDmw0wEG4Iot84gMwDkAzgrEYMgdjiHx99zMu1vkEEL8eY/CvmH71Nb9SsxnidWTC8esXi9/Gfr+Lyfh+f2DaXZmOT1gah7esIUZoCMhIBAwHAg0YeflSC+LFwPZw75jmD7HoNkjiaMmiTCFvH4D+JiJDInHVEhAN02QipAFdzps4QKR08C9lcnz/+niwnASgoOsyCen6p/eX/CjZ0mW+Dwz48fJvPPEn90Pw/j8QgAn/r2IwHCxyDny2dMbDw/IJZzx52MGsKkau4U5hvhePF8BmcUwMdcAfzp2zs9cn9lcSr+DCxuvMxNsrznEwNMdwjsO+qKpoXjrXtJiLJS/fwi+tzqF+1huIK+fo+YMcDeZ0RdIAEHYU6AsWzmczHjvkvln9NFIDF2VuW4RJPsSR/F8LNf3LYSsM0tYkxTn9jBcOz1zoNJKuwesTsinHKkRzjprLVpHnzxYU/dU0E6c15CUw/dFw12l977lw9yQP1O8e2xCp3zgpp3r1sv6zrFJybeUdOOGMmoecocmAO5ZXxqjRKfrB/Igx3BxbcMOW9O9xYpJyWxDFCZhcSPzBEGRdjxwORn38vZX8uetScluQRQ50t7ijL6lKHQyvsuWSizhlgpQjsTV6JuzbmqUZj4fU55bWAuJXF8e1zoIZ6vPqZ18bU2T5bmnixKnRUsIIVvTHzxdgU1/88h4Ud6ttBamTYsqmXSGINCBEwKJGnUpMe+tgjTCVI9nJOc7F2Xjo/PL6r6cAvcRGanr5zA6qvqoEb12zDNYUcUMCMw8OPyDw/vAIXB4SrGkzR7NVeKyb8+FkiiJ7MJ3sEgLP2mpZ4YZWIi7C7O6oLJNwRblVh2vRIioFS6iWBk5QXXUBpHquv1HnOomFA8kbQAT08olobai/wZAL4VQnedSXkymfJzm8/Lnfq/bnNyAHWyFzHjLKMlhufJc6IIC9Yi+w4jFm/YIicsSHwuENag2w3+tv9VZ6WzjGHf0A9FtoatfObLpnTfrIJDlF4U9OS5PbiuBOSPqB7cMpZ+ZSQf3V75l1SWHWLhAALMjCESqCmERsLwN7RPgwORE2NN4iz+dd5YDN/wliFNLk30gWPcTWcInVtf1C/IeIgjPRkbVlynvIIalSwZYO6PzG0CUGFytTblsMuyNp0W+WK4WHbdYbJ5OGRpCdlcjhNqdI5JQ3mvj1eOCWE9pjcG7PVWKb7cpGOBuVmySHlRtLTMxpa097aF8f2tP+yJw2E7fuCKLRGAANeEo69+WltvJw1aksXgwSdNvxepQ8cIyQgK9U27DkCOMl+PtzkywCcEUixw+Woj4MnErtQ5TFN/xP3T3qFCUj4dySJ1wocrQA6roMV2YUyBQegaKtl6E0wSwfDpoYOKyKhatv4xxTLODtz2afWEjvuWdWmf3dy6tb9uqcojjqzIOIhvnlC+kDHg6GVLiIwqMMEvLHP34gSlAR6vtYMhpLBgEHbIQMMpqY9nD39bO4WDQ4DaputbLHpbg0SZ0cx8V28ljHhQhmcFdEhWTy+mUVhUGvBPyW7I4xcEsfwH8BHxjNkwKOFi/JHRyCmS4ecUpFybLkYvRsyAKRuQhQeq/uQzkCI3OGmsdd1MS4bB6rIl4ZjsHIjGBxQCtjML+78mNPnm77iwzMi+uxo6JRXpJSLKFdrppq4MNKVYN2Wo2XRUJvPqo8k5tzHlKQH1FvrAUtRdSAH5cIHIrYUQKHGHveekUj74GAXEweyvIBV9Qb852jzF2DMnOEsyo2zy++dNAEQ5acL74rUF5YzMZL2awlXRlP4oSflPlbWLht4cp2+1Puy3bRnRogx8NIkoHUTnXoNE1Zkr0wXmyJLpXS2UW5WLvG5r+7Qvjy+OXOayWDSh2gBQC4b1mKZStaJaGTFwvrbFBbaI2T5DJ5oEDPVKmZGyeKjW2c92LHRK0e7rfgLRO0mRkxjBUssTaE32ifeSFYIyI2HApe1GJaaSHXCjByFSn57//8L7jCvEpSVLzl8gbBWKvsXrwmwbAZv/aCwwpIGFvBarKIMR8jxJS8+/jZ2pgnCQO3lh8Rbeq1VH/aGw4H47nbHXq9Xrc7mg5te2RPhr0p7Q7oeN51+4PJyLJ68747hY5edzKkfm8+8cfeqMem9sC25y6wOpsPBp7f35gXkXhr+RDZzl9BDfgrqIGwI7QkSKWqMyfHm+Yok/5OpY/RcRlvRyWlS3spKyoCNJcfO4o3GyJB+62J6x5NlZSe0Yh1bRXs2wpEK8EmrtRl2gTaUY+zpMOTmQRLflzYC3AaVvZgaBU03vp6Qa6+lgtsk3WkguNT7qDb3Z5uSeujy/beW4CcIT/LCnkRHMSP9/h+JODaDg51LTRoRXhrdb3i16zECVx4Y7hTt9rKa5HgTWJZIi5ayDctgV5Zf0m/SQAVpVeFpifj69bUpUi1uEBin90GHhayYM4Cy0llCERQ1x500UQ6tAcjszfaSF/N1cFEaYO9I5JdpRdXGvt9v+FT2Di8Qg71Lwggj5vvjUaRTCX9m0yUcqG+FjYtxc1kphUL8wFI5QRlZ7WQU+r5pspNB+OextOTwjebiRPBiBeeD9hymX6kLVVxXCpXknh3PTZRaRdtpkqL17TUKa/dJl8xL/eVZLLAlxd38+uGxyXNT16iDNKHj8BMoonGOmgSLfkCo9+cX4ghFnmtClIJnQMTcNmg5EJR/Pz1y7HgtjU45ktNYQn7FvWkxJiS9QqkOqPgfbigyebAa1mHRbBY1HrqHaPIDizi0C9gcXkq9C2/CagEQQPegSsSVESHKGdX3IOEEY/jdNbB507g+71qTtHwcuD4RjFPp1wJrMhXac/Zp/M97KNKsnlmVx6UsBTQ3yoKzNuEV3s381V5vcUKC9O1vIqKmPuXLUqThKWQMnAvwJPvDAw8EWHVtn/UBBiXgUMepTm0x30tntAkApEn/nLHoqOfQ7qkHcwXc2thTpdBGLC0zQs6Q7SThJ6LiXjhePVViG+xmK9fNPtqIW2xYu8g4wFnygvnGSylqJEo8Zp8OiEYDm5y8v8M97/LcPw1Eil4ThwQnkz7x2rIgfsB+ByDnLxCTsU/zXqCS/ZqfIzgcj4WAbHeZIrFPP3uUI92KKHJzcBN+hx/Pi5oyj6Kan6wX0QJOqCQBdzKvshfyZcZSDy/xco96zahS8cFQXsjiquE68rfZoUOL+g2yc2t41HwRtTn1EuQauJj+0lnT/w549Av8+cKj2R/aLDwjr7ww+ZlVxRvdROVbtxSPQr6nZus7aoOUea6cpjNIyqImgeVsDeMea6XoZZScqKshymrxuKJvfzK5Q9R3vLn5cLYcOTjEM4uOTCQdq/dlKEdyh1b9AGw/K+FZsxSPCD3W6puCzoogZsZMyzoWMVJJizWnnAI+j1bTxzvze2ysB8vl7ZPjcf5zRRz9RPqNBQFY84EL+VMVi2gzAe7Qn1UhDPzN+rLWI1tfKFSaS0BL3eVMVVK0xXacrNcQ6WxWJB2ycvqsW/L6qSxliLdm96CZbhLY217+WjuM03LYe0zvCnNVd5/87xawk5LIFVrHwK4Fi/zMofjErefR7zyLsnA5Qn59zKEwY36fgWwia/SYOl/4V0zgq9puCEAjox/RxMmEkl2Hys0DwcD/OqMjYE99RCD1Z6qygeF5YcLuYV9QvAhzoq6YL5kYMDghVad2iSsRM/nVO4Ck28ULfFw22iJOKm8s2gvq5km91XE3U/XnsdEHu2wCVpdTP1m+OUHADJMx7gYM7R6ic0jpQO1bTAr1RTvxBipmvZ6jUunea6kMcxlSYIuaKvY6xz2T7yQUfRWVLyOiuJJ4f3XyKwA18lcR9XwsOS7cG/WLlrkYlM8UdTH1kKKebOMKvr9ydzt2qPxiNLepNtz3dHI7vl0NHa9CXPp1Ov6vdGwb1nD7thm/pT1p1MPhBCj3dFkPHB9Npj0aH8yZYP5eDAZ0o1RxQJ1LbBYdHG3AmMrmIbiMRbNsVitz/ib/MeDJnWNGQxkJBR2/LtoNg3ieTHrTkk4CyiPKTZyckKMHvm9zETypyngA+wNRiTqEAr+EpOP6zxQFOvIifzbCEq5smZzBFDeWBuzqGLREq2pLQHMAvlXeyPc6lZEmkQviarNzCume1NMdaElgvHTuwTrCrEiW1VA++IrXMBfQzN5xcAxagL29vK1SFWmmfhaIDBxgswi70WtKELANKfIsyWMf5ENVqI2wYpx5F2Qso6PYOb9Xh5AS0HgpUgPNOqOUvBYUNPECWz3eCM85CpR6IxfX7NectQxSB2eR8Acq8z/sYCnYhEJCdJGWEigDA/qGtxbzEIGGGCWKR6uxtB064AnpV4jqiSj1cxLBirrb6moTeEbQt8INU+ZyzbYysZqzoeYBP5Yrxp4ZPMDKDUfscMvibwBQsma1YjBZUReIuLClm+Icdaz+jNZglyEwOMVFhaffzhtBOTyTE3+nUXIiZyW3mId3YDBgAUybUQnY7c2rze2B72J2R/vLWCe683PVa3FrVJ0DF8YmXO9NsnFpfPus1lVUJqBULn8mwi974HKp734BVK8XrBvO68/fDj//OHt6U8zURGJgePZ7Fx8FUilpfxNCk0/Naj8RTJ/4t8QPKipyS3vLuvvMGVVIJZDyffyNFvwZ2kygd0Rj0qEMOwU8W3+Zq0oo1CVcEHpE5ZUbHlFWf0R4LhsBaG6ffw2Aj5v65Q5iDcd+cidpzOB87FsDEUp+WD3JuKqpBZ5x7/wId0Kjj9UwTgMGMjhnIdugXtcBmZuim8xiv67RQziEDFtBSiiy18FOb4ea3pASUcXv05O1AOBdQMYt8LjOkPIvr9MCEi64FZIPvWNRnkhgZLD1mZ4PNbFaXWySa3lLrgyjK/X4DmgQeyC4GhtOx0ucMEp+pY68XyeYiU4ukgo+GQDyN7tnCFLELIgn5JuVt7t3VyZ7yI/BvH9d8AzyCpHfwbtR8PgOmpwGkohEWuPUuAdl5Y/Vtg+hJ+N9S21/FWW7DVWqJX9x0vY5JCUDuo7UYnp+snugCD04M4xAsWOcVKl7x60F7Qyd20fq9mN2wfuAa/29r9iUTTqvo0IOXeiHjcOyG4e3DrkRtZDDoYjYQZMp+b032cFsDDjZsDF+5+3GgI3pRq63LJv/yssAt1cpyFa1yCXV6swKMoZeE5vidapD3LY6Vq7Dm/LejeS/vvOXMilNAA/Z7kOHeVSIEoFRYa1djBFwRji4KcTcwIHPxz04fe+B48/3S2spd+dbSKAm9vNXKAJks4/I0R2ChBdeGxbK1hUf98yItq+YZQA/wNgIE42R1oAAA=="""
MODEL_URL = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q8_0.gguf?download=true"
MODEL_BYTES = 675710816
MODEL_SHA256 = "ca59ca7f13d0e15a8cfa77bd17e65d24f6844b554a7b6c12e07a5f89ff76844e"
ROOT = Path("/tmp/wave131")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
TARGET = ROOT / "target"
MODEL = ROOT / "qwen2.5-0.5b-instruct-q8_0.gguf"
FINAL_ZIP = Path("/kaggle/working/glcuda-t4-wave131-fused-swiglu-production-results.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)
RESULTS.mkdir(parents=True)

def run(cmd, *, cwd=None, env=None, timeout=14400, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({k: str(v) for k, v in env.items()})
    p = subprocess.run([str(x) for x in cmd], cwd=cwd, env=merged, text=True,
                       capture_output=True, timeout=timeout)
    print("$", " ".join(str(x) for x in cmd), flush=True)
    if p.stdout:
        print(p.stdout[-12000:], flush=True)
    if p.stderr:
        print(p.stderr[-12000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p

def save(name, p):
    (RESULTS / name).write_text(
        f"RETURN_CODE {p.returncode}\n\nSTDOUT\n{p.stdout}\n\nSTDERR\n{p.stderr}",
        encoding="utf-8",
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    digest = sha256_file(FINAL_ZIP)
    print("ARCHIVE", FINAL_ZIP, digest, flush=True)
    return digest

def percentile(values, q):
    values = sorted(values)
    x = (len(values) - 1) * q
    lo, hi = math.floor(x), math.ceil(x)
    return values[lo] if lo == hi else values[lo] * (hi - x) + values[hi] * (x - lo)

def bootstrap_ci(values, seed=118, draws=20000):
    rng = random.Random(seed)
    n = len(values)
    medians = [statistics.median(values[rng.randrange(n)] for _ in range(n))
               for _ in range(draws)]
    return [percentile(medians, 0.025), percentile(medians, 0.975)]

phase = "bootstrap"
try:
    embedded = gzip.decompress(base64.b64decode(PATCH_GZIP_B64))
    if hashlib.sha256(embedded).hexdigest() != PATCH_SHA256:
        raise RuntimeError("embedded patch hash mismatch")
    patch_path = RESULTS / "wave131.patch"
    patch_path.write_bytes(embedded)
    (RESULTS / "source.json").write_text(json.dumps({
        "build": BUILD, "base_rev": BASE_REV, "source_rev": SOURCE_REV,
        "patch_sha256": PATCH_SHA256, "patch_bytes": len(embedded),
    }, indent=2), encoding="utf-8")

    gpu = run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
               "--format=csv,noheader,nounits"], timeout=60)
    save("nvidia-smi.log", gpu)
    fields = [x.strip() for x in gpu.stdout.splitlines()[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"requires Tesla T4 sm_75, got {fields}")

    phase = "reconstruct"
    clone = run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800)
    save("git-clone.log", clone)
    checkout = run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=600)
    save("git-checkout.log", checkout)
    applied = run(["git", "apply", "--whitespace=error", patch_path], cwd=TREE)
    save("git-apply.log", applied)
    diff = run(["git", "diff", "--check"], cwd=TREE)
    save("git-diff-check.log", diff)

    cargo_candidates = [shutil.which("cargo"), Path.home() / ".cargo/bin/cargo",
                        "/usr/local/cargo/bin/cargo", "/opt/conda/bin/cargo"]
    cargo = next((str(x) for x in cargo_candidates if x and Path(x).is_file()), None)
    cargo_env = {}
    bootstrapped = False
    if cargo is None:
        bootstrapped = True
        rustup_script = ROOT / "rustup-init.sh"
        urllib.request.urlretrieve("https://sh.rustup.rs", rustup_script)
        cargo_home = ROOT / "cargo-home"
        rustup_home = ROOT / "rustup-home"
        cargo_env = {"CARGO_HOME": cargo_home, "RUSTUP_HOME": rustup_home}
        install = run(["bash", rustup_script, "-y", "--profile", "minimal",
                       "--default-toolchain", "stable", "--no-modify-path"],
                      env=cargo_env, timeout=1800)
        save("rustup-install.log", install)
        cargo = str(cargo_home / "bin/cargo")
    if not Path(cargo).is_file():
        raise RuntimeError(f"cargo unavailable after bootstrap: {cargo}")
    (RESULTS / "cargo-discovery.json").write_text(json.dumps({
        "selected": cargo, "bootstrapped": bootstrapped,
        "candidates": [str(x) for x in cargo_candidates if x],
    }, indent=2), encoding="utf-8")
    common = {**cargo_env, "CARGO_TARGET_DIR": TARGET, "CUDA_VISIBLE_DEVICES": "0"}

    phase = "host-tests"
    tests = run([cargo, "test", "-p", "glcuda", "--lib", "--locked"], cwd=TREE, env=common)
    save("cargo-lib-tests.log", tests)
    if "73 passed" not in tests.stdout or "0 failed" not in tests.stdout:
        raise RuntimeError("unexpected host test summary")

    phase = "cuda-parity"
    parity = run([cargo, "test", "--release", "-p", "glcuda", "--test", "parity",
                  "--locked", "--", "--nocapture", "--test-threads=1"],
                 cwd=TREE, env=common, check=False)
    save("cargo-cuda-parity.log", parity)
    if parity.returncode or "0 failed" not in parity.stdout:
        raise RuntimeError("CUDA parity failed")

    phase = "compiler-resource"
    ptxas = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    resource = run([
        ptxas, "-v", "-arch=sm_75",
        TREE / "glcuda/src/kernels/glcuda_sm75_wave129.ptx",
        "-o", ROOT / "wave131.cubin",
    ], check=False)
    save("ptxas-wave129.log", resource)
    resource_log = resource.stdout + "\n" + resource.stderr
    expected_resource = [
        "Used 80 registers", "used 1 barriers", "16384 bytes smem",
        "0 bytes stack frame", "0 bytes spill stores", "0 bytes spill loads",
    ]
    if resource.returncode or any(item not in resource_log for item in expected_resource):
        raise RuntimeError(f"Wave 131 resource gate failed: {resource_log[-4000:]}")

    phase = "model"
    urllib.request.urlretrieve(MODEL_URL, MODEL)
    model_meta = {"bytes": MODEL.stat().st_size, "sha256": sha256_file(MODEL)}
    if model_meta != {"bytes": MODEL_BYTES, "sha256": MODEL_SHA256}:
        raise RuntimeError(f"model identity mismatch: {model_meta}")
    (RESULTS / "model.json").write_text(json.dumps(model_meta, indent=2), encoding="utf-8")

    phase = "build"
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave131_fused_swiglu_production", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)
    exe = TARGET / "release/examples/wave131_fused_swiglu_production"
    prod_env = {**common, "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
                "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_Q8_NOSTORE": "1", "GLCUDA_FFN_GATE_UP_STACKED": "1", "GLCUDA_DEFER_FFN_RESIDUAL": "1", "GLCUDA_NTILE128": "1",
                "GLCUDA_BSTAGE": "1", "GLCUDA_GEMM_N16": "1",
                "GLCUDA_GEMM_N16_PREFETCH": "1", "GLCUDA_N16_FUSED_SWIGLU": "1", "GLCUDA_ATTN_MMA4": "1",
                "GLCUDA_ATTN_MMA4_REGQ": "1", "GLCUDA_ATTN_MMA4_AV": "1"}

    phase = "production-ab"
    records = []
    summaries = []
    for invocation in ["a", "b"]:
        measured = run([exe, MODEL, invocation], cwd=TREE, env=prod_env, check=False)
        save(f"production-{invocation}.log", measured)
        if measured.returncode:
            raise RuntimeError(f"production {invocation} failed")
        for raw in re.findall(r"\[wave131-sample\]\s*(\{[^\n]+\})", measured.stdout):
            records.append(json.loads(raw))
        found = re.search(r"\[wave131-summary\]\s*(\{[^\n]+\})", measured.stdout)
        if not found:
            raise RuntimeError(f"production {invocation} summary missing")
        summaries.append(json.loads(found.group(1)))
    if len(records) != 80:
        raise RuntimeError(f"expected 80 production samples, saw {len(records)}")
    if {r["oracle_token"] for r in records} != {3323}:
        raise RuntimeError("oracle drift in production samples")

    retained = [r["prefill_ms"] for r in records if r["arm"] == "retained"]
    candidate = [r["prefill_ms"] for r in records if r["arm"] == "candidate"]
    if len(retained) != len(candidate) or len(retained) != 40:
        raise RuntimeError(f"bad arm sample counts: retained={len(retained)} candidate={len(candidate)}")
    retained_sorted = sorted(retained)
    candidate_sorted = sorted(candidate)
    retained_median = retained_sorted[len(retained_sorted) // 2]
    candidate_median = candidate_sorted[len(candidate_sorted) // 2]

    phase = "candidate-profile"
    prof_env = {**prod_env, "GLCUDA_TELEMETRY": "1"}
    prof = run([exe, MODEL, "profile"], cwd=TREE, env=prof_env, check=False)
    save("candidate-profile.log", prof)
    if prof.returncode or "[wave131-profile]" not in prof.stdout:
        raise RuntimeError("candidate production profile failed")
    profile = json.loads(re.search(r"\[wave131-profile\]\s*(\{[^\n]+\})", prof.stdout).group(1))
    stages = [json.loads(x) for x in re.findall(r"\[wave131-stage\]\s*(\{[^\n]+\})", prof.stdout)]
    expected_names = [
        "qkv",
        "attn_norm",
        "attn_kv_write",
        "attention",
        "attn_out_quant",
        "ffn_down",
        "ffn_gate_up",
        "attn_out",
        "lm_head",
        "ffn_residual_norm_quant",
        "ffn_silu_quant",
        "ffn_residual_add",
    ]
    if [s["name"] for s in stages] != expected_names or profile["oracle_token"] != 3323:
        raise RuntimeError(f"profile contract failed: {profile}, {stages}")
    stage_sum = sum(x["total_ms"] for x in stages)
    ranked = sorted(
        [
            {**s,
             "share_of_gpu_total": s["total_ms"] / profile["gpu_prefill_ms"],
             "share_of_stage_sum": s["total_ms"] / stage_sum if stage_sum else 0.0}
            for s in stages
        ],
        key=lambda s: -s["total_ms"],
    )
    by_name = {x["name"]: x for x in stages}
    phase = "glbench-ceiling"
    glbench_build = run([cargo, "build", "--release", "-p", "glbench", "--locked"],
                        cwd=TREE, env=common)
    save("glbench-build.log", glbench_build)
    glbench = TARGET / "release/glbench"
    ceiling_prompt = ("Measure this deterministic systems prompt carefully. Explain how "
                      "token-parallel integer matrix multiplication uses shared memory, "
                      "Tensor Cores, and fixed launch geometry. " * 8)
    base_env = dict(prod_env)
    base_env.pop("GLCUDA_N16_FUSED_SWIGLU", None)
    ceiling_runs = []
    for index, arm in enumerate(["retained", "candidate", "candidate", "retained"]):
        out = RESULTS / f"glbench-{index + 1}-{arm}.json"
        env = prod_env if arm == "candidate" else base_env
        measured = run([
            glbench, "run", "--engine", "glcuda", "--model", MODEL,
            "--prompt", ceiling_prompt, "--tokens", "64", "--cold-iters", "1",
            "--warmup", "2", "--iters", "10", "--temperature", "0", "--seed", "42",
            "--kind", "end_to_end", "--out", out,
        ], cwd=TREE, env=env, check=False)
        save(f"glbench-{index + 1}-{arm}.log", measured)
        if measured.returncode or not out.is_file():
            raise RuntimeError(f"glbench ceiling run failed: {index + 1} {arm}")
        dispatch_log = measured.stdout + "\n" + measured.stderr
        fused_dispatched = '"path":"wave129-fused-swiglu-production"' in dispatch_log
        contract_enabled = (
            '"q8_nostore":true' in dispatch_log
            and '"ffn_gate_up_stacked":true' in dispatch_log
            and '"defer_ffn_residual":true' in dispatch_log
        )
        if not contract_enabled:
            raise RuntimeError(f"retained prerequisite contract missing: {index + 1} {arm}")
        if (arm == "candidate") != fused_dispatched:
            raise RuntimeError(
                f"fused dispatch mismatch: run={index + 1} arm={arm} dispatched={fused_dispatched}"
            )
        session = json.loads(out.read_text(encoding="utf-8"))
        analysis = session["analysis"]
        ceiling_runs.append({
            "order": index + 1,
            "arm": arm,
            "prefill_tps": analysis["prefill_tps"],
            "decode_tps": analysis["decode_tps"],
            "ceiling_efficiency": analysis.get("ceiling_efficiency"),
            "ceiling_basis": analysis.get("ceiling_basis"),
            "bottleneck": analysis.get("bottleneck"),
            "notes": analysis.get("notes", []),
        })

    summary = {
        "wave": 131,
        "candidate": "wave129_fused_swiglu_production",
        "gpu": fields,
        "model": model_meta,
        "glbench_ceiling_runs": ceiling_runs,
        "production_ab": {
            "samples_per_arm": len(retained),
            "retained_median_ms": retained_median,
            "candidate_median_ms": candidate_median,
            "retained_tps": 244000.0 / retained_median,
            "candidate_tps": 244000.0 / candidate_median,
            "speedup": retained_median / candidate_median,
            "all_candidate_deltas_positive": all(c < r for c, r in zip(candidate, retained)),
            "invocation_summaries": summaries,
        },
        "candidate_profile": profile,
        "candidate_stages": stages,
        "candidate_ranked_stages": ranked,
        "candidate_stage_sum_ms": stage_sum,
        "candidate_attention_ms": sum(by_name[n]["total_ms"] for n in [
            "qkv", "attn_norm", "attn_kv_write", "attention", "attn_out_quant", "attn_out"
        ]),
        "candidate_ffn_ms": sum(by_name[n]["total_ms"] for n in [
            "ffn_down", "ffn_gate_up", "ffn_residual_norm_quant", "ffn_silu_quant",
            "ffn_residual_add"
        ]),
        "retention_authority": True,
        "target_15000_tps_achieved": 244000.0 / candidate_median >= 15000,
    }
    report = [
        "# Wave 131 fused SwiGLU production and ceiling",
        "",
        f"- Production A/B samples: {len(retained)} per arm, counterbalanced a+b",
        f"- Retained median: {retained_median:.6f} ms = {summary['production_ab']['retained_tps']:.1f} tok/s",
        f"- Candidate median: {candidate_median:.6f} ms = {summary['production_ab']['candidate_tps']:.1f} tok/s",
        f"- Speedup: {summary['production_ab']['speedup']:.4f}x",
        f"- Target 15k reached: {summary['target_15000_tps_achieved']}",
        f"- Decode P50 range: {min(r['decode_tps']['median'] for r in ceiling_runs):.3f}.."
        f"{max(r['decode_tps']['median'] for r in ceiling_runs):.3f} tok/s",
        f"- Candidate profile GPU: {profile['gpu_prefill_ms']:.6f} ms = {profile['gpu_prefill_tps']:.1f} tok/s",
        "",
        "| candidate stage | ms | share of GPU total | calls | bytes read | macs |",
        "|---|---:|---:|---:|---:|---:|",
    ]
    for s in ranked:
        report.append(
            f"| `{s['name']}` | {s['total_ms']:.6f} | {100.0 * s['share_of_gpu_total']:.2f}% | "
            f"{s['calls']} | {s['bytes_read']} | {s['macs']} |"
        )
    (RESULTS / "production-records.json").write_text(json.dumps(records, indent=2), encoding="utf-8")
    (RESULTS / "wave131-summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    (RESULTS / "REPORT.md").write_text("\n".join(report), encoding="utf-8")
    print("WAVE131_RESULT", json.dumps(summary, indent=2), flush=True)
    archive()
except Exception:
    (RESULTS / "FAILED.json").write_text(json.dumps({
        "phase": phase, "traceback": traceback.format_exc()}, indent=2), encoding="utf-8")
    archive()
    raise
